# 5 Scaled Dot-Product Attention

**Queries, Keys, and Values:** $\quad$ $Q,K\in\mathbb{R}^{n\times d_{\text{k}}}\quad V\in\mathbb{R}^{n\times d_{\text{v}}}$
* Each query $\,\boldsymbol{q}_i=Q_{i,:}\,$ is a value token to be predicted.
* Each key-value pair $\,(\boldsymbol{k}_i,\boldsymbol{v}_i)=(K_{i,:},V_{i,:})\,$ is a token-value reference.
* We assume that similar tokens contribute informationally relevant values.

**Attention Scores:** $\;$ We use $\boldsymbol{q}_iK^t\in\mathbb{R}^{1\times n}\,$ to measure the **attention** that $\,\boldsymbol{q}_i\,$ must be given to each key
$$\operatorname{AttentionScores}(Q,K)=QK^t\in\mathbb{R}^{n\times n}$$

[**Property:**](https://funnywii.com/upload/proof.pdf) $\;$ if $\boldsymbol{q}_i^t$ and $\boldsymbol{k}_j^t$ are $\mathcal{N}_{d_{\text{k}}}(\boldsymbol{0},\mathbf{I}),\,$ then $\boldsymbol{q}_i\boldsymbol{k}_j^t$ follows a distribution with zero mean and variance $d_{\text{k}}$

**Scaled Attention Scores:** $\;$ $\operatorname{ScaledAttentionScores}(Q,K)%
=\frac{1}{\sqrt{d_{\text{k}}}}\operatorname{AttentionScores}(Q,K)\in\mathbb{R}^{n\times n}$

**Attention Weights:** $\;$ $\operatorname{AttentionWeights}(Q,K)%
=\operatorname{Softmax}\left(\operatorname{ScaledAttentionScores}(Q,K)\right)\in\mathbb{R}^{n\times n}$

**Scaled dot-product attention (see zoom-in 2 in architecture):** $\;$ The value of each query is obtained as the sum of the reference values ​​weighted by their attention weights corresponding
$$\operatorname{Attention}(Q,K,V)%
=\operatorname{AttentionWeights}(Q,K)V\in\mathbb{R}^{n\times d_{\text{v}}}$$

**Exercise:** $\;$ Compute $\,\operatorname{Attention}(Q,K,V)\,$ with
$\,Q=K=V=\begin{pmatrix}-0.7071&0.7071\\0.7071&-0.7071\\0.7070&-0.7070\end{pmatrix}$


<p style="page-break-after:always;"></p>


**Solution:**

$$\begin{align*}
\operatorname{AttentionScores}(Q,K)%
&=\begin{pmatrix}-0.7071&0.7071\\0.7071&-0.7071\\0.7070&-0.7070\end{pmatrix}%
\begin{pmatrix}-0.7071&0.7071&0.7070\\0.7071&-0.7071&-0.7070\end{pmatrix}\\%
&=\begin{pmatrix}1&-1&-1\\-1&1&1\\-1&1&1\end{pmatrix}
\end{align*}$$

$$\operatorname{ScaledAttentionScores}(Q,K)%
=\begin{pmatrix}0.7071&-0.7071&-0.7070\\-0.7071&0.7071&0.7070\\-0.7070&0.7070&0.7069\end{pmatrix}$$

$$\operatorname{AttentionWeights}(Q,K)%
=\begin{pmatrix}0.6728&0.1636&0.1636\\0.1084&0.4458&0.4458\\0.1084&0.4458&0.4458\end{pmatrix}$$

$$\operatorname{Attention}(Q,K,V)%
=\begin{pmatrix}0.6728&0.1636&0.1636\\0.1084&0.4458&0.4458\\0.1084&0.4458&0.4458\end{pmatrix}%
\begin{pmatrix}-0.7071&0.7071\\0.7071&-0.7071\\0.7070&-0.7070\end{pmatrix}%
=\begin{pmatrix}-0.2444&0.2444\\0.5538&-0.5538\\0.5538&-0.5538\end{pmatrix}$$


<p style="page-break-after:always;"></p>


In [1]:
import torch; import torch.nn as nn; torch.manual_seed(23)
import import_ipynb; from at251 import create_model
model = create_model(src_vocab_size=3, tgt_vocab_size=3, embed_dim=2, num_layers=1, num_heads=1, dropout=0.)
src = torch.LongTensor([[1, 2, 1]]); x = model.src_embed(src).data
norm_x = model.encoder.layers[0].norm_self_attn(x).data; norm_x

tensor([[[-0.7071,  0.7071],
         [ 0.7071, -0.7071],
         [ 0.7070, -0.7070]]])

In [2]:
query = key = value = norm_x; d_k = query.size(-1)
scores = torch.matmul(query, key.transpose(-2, -1)); scores

tensor([[[ 1.0000, -1.0000, -0.9999],
         [-1.0000,  1.0000,  0.9999],
         [-0.9999,  0.9999,  0.9998]]])

In [3]:
import math; scaled_scores = scores / math.sqrt(d_k); scaled_scores

tensor([[[ 0.7071, -0.7071, -0.7070],
         [-0.7071,  0.7071,  0.7070],
         [-0.7070,  0.7070,  0.7069]]])

In [4]:
p_attn = scaled_scores.softmax(dim=-1); p_attn

tensor([[[0.6728, 0.1636, 0.1636],
         [0.1084, 0.4458, 0.4458],
         [0.1084, 0.4458, 0.4458]]])

In [5]:
self_attn = torch.matmul(p_attn, value); self_attn

tensor([[[-0.2444,  0.2444],
         [ 0.5538, -0.5538],
         [ 0.5538, -0.5538]]])


<p style="page-break-after:always;"></p>
